# 02.2 - Mô hình Hybrid SVD (Matrix Factorization + User Preferences)

**Mục tiêu:**
Xây dựng mô hình Gợi ý dựa trên sự kết hợp giữa:
1. **SVD (Singular Value Decomposition):** Phân tích ma trận User-Item để tìm ra các đặc trưng ẩn (latent features) của người dùng và bài hát.
2. **User Preferences:** Lọc và tăng trọng số cho các bài hát phù hợp với thể loại (Genre) và ngôn ngữ (Language) yêu thích nhất của người dùng.
Mô hình sẽ được xuất ra file `.pkl` để có thể sử dụng trực tiếp trong ứng dụng thực tế.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.size'] = 12
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

# Tạo thư mục lưu model nếu chưa có
os.makedirs('models', exist_ok=True)

print('✅ Import thư viện thành công!')

✅ Import thư viện thành công!


## 1. Tải và Lọc Dữ liệu
Sử dụng file `merged_data.csv`. Tương tự KNN, để tối ưu hóa, chúng ta sẽ lọc bỏ những bài hát "đuôi dài" (long-tail) - tức là những bài hát có ít hơn 10 lượt nghe.

In [2]:
# Đọc dữ liệu
df = pd.read_csv('processed_data/merged_data.csv')
print(f'Kích thước dữ liệu gốc: {df.shape[0]:,} dòng x {df.shape[1]} cột')

# Tính số lượt nghe của mỗi bài hát
song_counts = df['song_id'].value_counts()

# Lọc các bài hát có ít nhất 10 lượt nghe
popular_songs = song_counts[song_counts >= 10].index
df_filtered = df[df['song_id'].isin(popular_songs)].reset_index(drop=True)

print(f'Số lượng bài hát ban đầu: {len(song_counts):,}')
print(f'Số lượng bài hát sau khi lọc (>=10 lượt nghe): {len(popular_songs):,}')
print(f'Kích thước dữ liệu sau khi lọc: {df_filtered.shape[0]:,} dòng')

# ===== THÊM DÒNG NÀY =====
# Tạo cột primary_genre từ genre_ids
if 'primary_genre' not in df_filtered.columns:
    df_filtered['primary_genre'] = df_filtered['genre_ids'].astype(str).str.split('|').str[0]
    print("Đã tạo cột primary_genre")

# Trích xuất thông tin bài hát
song_info_cols = ['song_id', 'song_name', 'artist_name', 'primary_genre', 'language']
song_details = df_filtered[song_info_cols].drop_duplicates(subset=['song_id']).set_index('song_id')

Kích thước dữ liệu gốc: 174,010 dòng x 16 cột
Số lượng bài hát ban đầu: 20,474
Số lượng bài hát sau khi lọc (>=10 lượt nghe): 2,866
Kích thước dữ liệu sau khi lọc: 136,326 dòng


## 2. Xây dựng Hồ sơ Sở thích (User Taste Profiles)
Phân tích lịch sử của từng người dùng để tìm ra **Thể loại (Genre)** và **Ngôn ngữ (Language)** mà họ nghe nhiều nhất.

In [3]:
# Tính toán Thể loại yêu thích nhất của từng user
user_top_genre = df_filtered.groupby('msno')['primary_genre'].agg(lambda x: x.value_counts().index[0]).reset_index()
user_top_genre.columns = ['msno', 'top_genre']

# Tính toán Ngôn ngữ yêu thích nhất của từng user
user_top_lang = df_filtered.groupby('msno')['language'].agg(lambda x: x.value_counts().index[0]).reset_index()
user_top_lang.columns = ['msno', 'top_language']

# Gộp thành User Profile
user_profiles = pd.merge(user_top_genre, user_top_lang, on='msno')
user_profiles.set_index('msno', inplace=True)

print(f'Đã tạo hồ sơ sở thích cho {len(user_profiles):,} người dùng.')
user_profiles.head()

Đã tạo hồ sơ sở thích cho 10,726 người dùng.


,top_genre,top_language
msno,,
++5wYjoMgQHoRuD3GbbvmphZbBBwymzv5Q4l8sywtuU=,458,3.0
++AH7m/EQ4iKe6wSlfO/xXAJx50p+fCeTyF90GoE9Pg=,465,3.0
++e+jsxuQ8UEnmW40od9Rq3rW7+wAum4wooXyZTKJpk=,465,3.0
+/UwoUi5+rNj/F6RO6gMrMhOy0oTzs90MWKVNZs4+Wg=,465,3.0
+0e12C+p9dzDbOvKjt8eElKH9yZPshAstxjm60XFgSM=,465,3.0


## 3. Xây dựng Ma trận Tương tác User-Item
Để chạy thuật toán SVD, ta cần ma trận thưa với cấu trúc:
- Hàng (Rows): Người dùng (`msno`)
- Cột (Columns): Bài hát (`song_id`)

In [4]:
# Tạo ID số nguyên cho User và Song để đưa vào ma trận thưa
user_ids = df_filtered['msno'].astype('category').cat.categories
song_ids = df_filtered['song_id'].astype('category').cat.categories

# Dictionary ánh xạ giữa Index và ID gốc
user_to_idx = {user: i for i, user in enumerate(user_ids)}
song_to_idx = {song: i for i, song in enumerate(song_ids)}
idx_to_user = {i: user for user, i in user_to_idx.items()}
idx_to_song = {i: song for song, i in song_to_idx.items()}

# Ánh xạ ID vào DataFrame
df_filtered['user_idx'] = df_filtered['msno'].map(user_to_idx)
df_filtered['song_idx'] = df_filtered['song_id'].map(song_to_idx)

# Trọng số tương tác = 1 (vì chỉ có target=1)
interactions = np.ones(len(df_filtered))

# Khởi tạo CSR Matrix: (Số người dùng x Số bài hát)
user_item_matrix = csr_matrix((interactions, (df_filtered['user_idx'], df_filtered['song_idx'])), 
                              shape=(len(user_ids), len(song_ids)))

print(f'Kích thước ma trận thưa User-Item: {user_item_matrix.shape}')

Kích thước ma trận thưa User-Item: (10726, 2866)


## 4. Huấn luyện Mô hình SVD
Sử dụng `scipy.sparse.linalg.svds` để phân tích ma trận thành các nhân tố ẩn.

In [5]:
# Chuyển đổi ma trận sparse sang kiểu float để tính SVD
user_item_matrix_float = user_item_matrix.asfptype()

# Áp dụng SVD để lấy K đặc trưng ẩn (latent factors)
# k: Số lượng features, càng cao càng chi tiết nhưng dễ overfit và chậm
K = 100
U, sigma, Vt = svds(user_item_matrix_float, k=K)

# Chuyển sigma thành ma trận đường chéo
sigma = np.diag(sigma)

print('✅ Huấn luyện mô hình SVD thành công!')
print(f'Kích thước ma trận U (User Features): {U.shape}')
print(f'Kích thước ma trận Sigma (Weights): {sigma.shape}')
print(f'Kích thước ma trận Vt (Item Features): {Vt.shape}')

✅ Huấn luyện mô hình SVD thành công!
Kích thước ma trận U (User Features): (10726, 100)
Kích thước ma trận Sigma (Weights): (100, 100)
Kích thước ma trận Vt (Item Features): (100, 2866)


## 5. Lưu Mô hình và Dữ liệu (Export to .pkl)
Chúng ta sẽ lưu các đối tượng cần thiết bằng thư viện `joblib` để sử dụng cho Web Demo sau này.

In [6]:
# Lưu các đối tượng vào thư mục models/
joblib.dump(U, 'models/svd_U.pkl')
joblib.dump(sigma, 'models/svd_sigma.pkl')
joblib.dump(Vt, 'models/svd_Vt.pkl')
joblib.dump(user_to_idx, 'models/svd_user_to_idx.pkl')
joblib.dump(song_to_idx, 'models/svd_song_to_idx.pkl')
joblib.dump(idx_to_song, 'models/svd_idx_to_song.pkl')
joblib.dump(user_profiles, 'models/user_profiles.pkl')

# Lưu cấu trúc lịch sử người dùng để kiểm tra bài đã nghe
user_history = df_filtered.groupby('msno')['song_id'].apply(set).to_dict()
joblib.dump(user_history, 'models/svd_user_history.pkl')

print('💾 Đã lưu tất cả files mô hình SVD vào thư mục models/')

💾 Đã lưu tất cả files mô hình SVD vào thư mục models/


## 6. Xây dựng Hàm Gợi ý Hybrid
Hàm kết hợp:
1. Tính điểm gợi ý cho toàn bộ bài hát từ ma trận phân rã SVD.
2. Bỏ qua bài hát User đã nghe, lấy top các bài hát điểm cao nhất.
3. Boost Score: Cộng thêm điểm nếu bài hát có Genre hoặc Language khớp với `User Taste Profile`.

In [7]:
def recommend_svd_hybrid(user_id, top_n=10, genre_boost=0.5, lang_boost=0.8):
    if user_id not in user_to_idx:
        return "User không tồn tại trong tập huấn luyện."
        
    u_idx = user_to_idx[user_id]
    
    # 1. Tính điểm dự đoán của user này cho TẤT CẢ bài hát
    # Phép nhân: U[u_idx, :] * sigma * Vt
    user_ratings = np.dot(np.dot(U[u_idx, :], sigma), Vt)
    
    # 2. Lấy danh sách các bài user đã nghe
    listened_songs = user_history.get(user_id, set())
    
    candidate_songs = {}
    
    # Lấy top các bài hát điểm cao nhất từ SVD (lấy nhiều hơn top_n để dự phòng boost)
    top_song_indices = np.argsort(user_ratings)[::-1]
    
    for s_idx in top_song_indices:
        song_id = idx_to_song[s_idx]
        
        # Bỏ qua bài hát user đã từng nghe
        if song_id in listened_songs:
            continue
            
        candidate_songs[song_id] = user_ratings[s_idx]
        if len(candidate_songs) >= top_n * 5:
            break
            
    if not candidate_songs:
        return "Không có gợi ý phù hợp."

    # 3. Chấm điểm ưu tiên theo User Taste Profile
    user_taste = user_profiles.loc[user_id]
    user_pref_genre = user_taste['top_genre']
    user_pref_lang = user_taste['top_language']
    
    # Max rating để chuẩn hóa điểm boost
    max_rating = max(candidate_songs.values())
    if max_rating <= 0: max_rating = 1.0
    
    scored_candidates = []
    for song_id, base_score in candidate_songs.items():
        song_info = song_details.loc[song_id]
        final_score = base_score
        
        # Tăng điểm nếu khớp Thể loại
        if song_info['primary_genre'] == user_pref_genre:
            final_score += genre_boost * max_rating
            
        # Tăng điểm nếu khớp Ngôn ngữ
        if song_info['language'] == user_pref_lang:
            final_score += lang_boost * max_rating
            
        scored_candidates.append((song_id, final_score))
        
    # 4. Xếp hạng và lấy Top N
    scored_candidates.sort(key=lambda x: x[1], reverse=True)
    top_song_ids = [item[0] for item in scored_candidates[:top_n]]
    
    # 5. Định dạng kết quả
    result_df = song_details.loc[top_song_ids].reset_index()
    result_df['hybrid_score'] = [item[1] for item in scored_candidates[:top_n]]
    
    return result_df[['song_id', 'song_name', 'artist_name', 'primary_genre', 'language', 'hybrid_score']]

print('✅ Đã tạo hàm recommend_svd_hybrid')

✅ Đã tạo hàm recommend_svd_hybrid


## 7. Demo & Kiểm thử Gợi ý
Chạy thử mô hình với một số người dùng ngẫu nhiên.

In [8]:
# Lấy ngẫu nhiên 1 User ID
sample_user = user_profiles.sample(1, random_state=42).index[0]

print(f"👤 Gợi ý cho User: {sample_user}")
print("-" * 50)
print(f"Hồ sơ Sở thích (Taste Profile):")
print(f" - Thể loại yêu thích: {user_profiles.loc[sample_user]['top_genre']}")
print(f" - Ngôn ngữ yêu thích: {user_profiles.loc[sample_user]['top_language']}")

print("\n🎵 Top 10 bài hát gợi ý (SVD + User Preferences):")
recs = recommend_svd_hybrid(sample_user, top_n=10)
display(recs)

👤 Gợi ý cho User: 3xBCNT7BqgMGbebWUZRtvcEYYKwEbDfWAEAvDrZ5lV8=
--------------------------------------------------
Hồ sơ Sở thích (Taste Profile):
 - Thể loại yêu thích: 465.0
 - Ngôn ngữ yêu thích: 3.0

🎵 Top 10 bài hát gợi ý (SVD + User Preferences):


,song_id,song_name,artist_name,primary_genre,language,hybrid_score
0,rFLawM8j/vW78o/S3P2GTsmy6HwkTI15P7NF/i9CkX4=,以後以後 (A better tomorrow),林芯儀 (Shennio Lin),465,3.0,0.199602
1,7NV5ZpaFlc7e3fjuxNiiuL0+HEr8qASFr+HHztIhwGY=,幸福了 然後呢,A-Lin,465,3.0,0.185217
2,jFUlstiDM/0JBpQz1N1Mr1uJj0H8kiQsJjfbnx45gHs=,信愛成癮 (Love addiction),陳嘉樺 (Ella Chen),465,3.0,0.174872
3,8qWeDv6RTv+hYJxW94e7n6HBzHPGPEZW9FuGhj6pPhQ=,愛．這件事情,傅又宣 (Maggie Fu),465,3.0,0.174622
4,GFWjz4apE8zMiXe6wc0qCjr+DbK9BFkwo/rr+FsNm+g=,癡情男子漢,玖壹壹,465,3.0,0.168957
5,iiifj//EcKnociT6W3nBASJygGe/TyFB41UD1bTtnU0=,失落沙洲,徐佳瑩 (Lala Hsu),465,3.0,0.167177
6,At7ayZb8zrkdvXw+EBt7z5e4IZRe0JhfOC8HqzzA0sg=,平凡之路,朴樹 (Pu Shu),465,3.0,0.165700
7,ZcKgNis1AP1LA0sdtIddrtk7P04iiJzJrXvwXdT/X3Q=,妮妮 (Nini),那對夫妻 (Nico&Kim),458,3.0,0.162176
8,TIwOs7iFTKo3Cy2yiNReYYcZc1JyAx+0k08+z97k1dA=,有點甜,汪蘇瀧 (Silence Wang),465,3.0,0.161034
9,B+lDDFfdRD9WVY2vdfJX3DXLlKohkLJC+mxE+Hfnh/o=,偷偷的【三立華劇[幸福兌換券]插曲】,MP魔幻力量 (Magic Power),465,3.0,0.160535


## 8. Đánh giá (Evaluation - Hit Rate@10)
Chúng ta sẽ kiểm tra xem với mô hình mới này, tỷ lệ Hit Rate như thế nào.

In [9]:
import random

def evaluate_hit_rate_svd_hybrid(test_users, top_n=10):
    hits = 0
    total = 0
    
    for user in test_users:
        if user not in user_history:
            continue
        user_songs = list(user_history[user])
        if len(user_songs) < 2:
            continue
            
        # Ẩn 1 bài hát
        hidden_song = random.choice(user_songs)
        
        # Sửa lại lịch sử tạm thời
        user_history[user].remove(hidden_song)
        
        # Gợi ý
        recs = recommend_svd_hybrid(user, top_n=top_n)
        
        # Phục hồi lịch sử
        user_history[user].add(hidden_song)
        
        if type(recs) != str and hidden_song in recs['song_id'].values:
            hits += 1
        total += 1
        
    hit_rate = hits / total if total > 0 else 0
    return hit_rate, total

# Lấy 100 users có >= 2 bài hát để test
eligible_users = [u for u, songs in user_history.items() if len(songs) >= 2]
test_users_sample = random.sample(eligible_users, min(100, len(eligible_users)))

print(f"Đang đánh giá Hit Rate@{10} trên {len(test_users_sample)} users...")
hit_rate, total_tested = evaluate_hit_rate_svd_hybrid(test_users_sample, top_n=10)

print(f"✅ Đánh giá hoàn tất!")
print(f"Hit Rate @ 10: {hit_rate:.2%} (Trên tổng {total_tested} test cases)")

Đang đánh giá Hit Rate@10 trên 100 users...
✅ Đánh giá hoàn tất!
Hit Rate @ 10: 41.00% (Trên tổng 100 test cases)
